# Vertex AI Search Indexing

This notebook builds the Vertex AI Search data store used by tutorial 10, `t10_vertex_search_agent`.

You will:

1. set a PDF URL;
2. download the PDF locally;
3. upload the PDF to a Google Cloud Storage staging bucket;
4. create or reuse a Vertex AI Search data store;
5. import the PDF from GCS into the data store;
6. print the data store ID to copy into `.env`.

## Prerequisites

```bash
uv sync --extra adk --extra notebooks --extra rag
gcloud auth application-default login
```

Enable the required GCP APIs (one-time per project):

```bash
gcloud services enable \
  discoveryengine.googleapis.com \
  storage.googleapis.com \
  --project=YOUR_PROJECT_ID
```

Required `.env` values: `GOOGLE_CLOUD_PROJECT` and `VERTEX_RAG_STAGING_BUCKET` (any existing GCS bucket).

In [1]:
from pathlib import Path
import os


DEFAULT_PDF_URL = (
    "https://sede.agenciatributaria.gob.es/static_files/Sede/Biblioteca/Manual/"
    "Practicos/IRPF/IRPF-2025/ManualRenta2025Parte1_es_es.pdf"
)


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("Could not find repository root")


def load_env_file(env_file: Path) -> None:
    if not env_file.exists():
        print("No .env file found; expecting configuration in the environment")
        return

    for line in env_file.read_text().splitlines():
        line = line.strip()
        if "=" in line and not line.startswith("#"):
            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip().strip('"').strip("'")
    print(f"Loaded {env_file.relative_to(REPO_ROOT)}")


REPO_ROOT = find_repo_root(Path.cwd())
ENV_FILE = REPO_ROOT / ".env"
load_env_file(ENV_FILE)

TUTORIAL_DIR = REPO_ROOT / "src" / "tutorials" / "t10_vertex_search_agent"
DOWNLOAD_DIR = TUTORIAL_DIR / "downloads"
PDF_URL = os.getenv("VERTEX_SEARCH_PDF_URL") or os.getenv("LOCAL_RAG_PDF_URL", DEFAULT_PDF_URL)
PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT", "").strip()
STAGING_BUCKET = os.getenv("VERTEX_RAG_STAGING_BUCKET", "").strip()
DATA_STORE_DISPLAY_NAME = os.getenv("VERTEX_SEARCH_DATA_STORE_NAME", "agents-tutorials-search")
DATA_STORE_SLUG = DATA_STORE_DISPLAY_NAME.lower().replace(" ", "-")
EXISTING_DATA_STORE_ID = os.getenv("VERTEX_SEARCH_DATA_STORE_ID", "").strip()

if not PROJECT_ID:
    raise ValueError("GOOGLE_CLOUD_PROJECT is required")
if not STAGING_BUCKET:
    raise ValueError("VERTEX_RAG_STAGING_BUCKET is required (e.g. gs://your-bucket)")

print(f"PDF URL:        {PDF_URL}")
print(f"Project:        {PROJECT_ID}")
print(f"Staging bucket: {STAGING_BUCKET}")
print(f"Data store:     {EXISTING_DATA_STORE_ID or f'(create {DATA_STORE_SLUG!r})'}")

Loaded .env
PDF URL:        https://sede.agenciatributaria.gob.es/static_files/Sede/Biblioteca/Manual/Practicos/IRPF/IRPF-2025/ManualRenta2025Parte1_es_es.pdf
Project:        edem-practica-mlops
Staging bucket: gs://edem-practica-mlops-agent-staging
Data store:     (create 'agents-tutorials-search')


## 1. Download The PDF

In [2]:
from urllib.parse import unquote, urlparse
from urllib.request import urlretrieve


def filename_from_url(url: str) -> str:
    path = unquote(urlparse(url).path)
    filename = Path(path).name or "source.pdf"
    return filename if filename.lower().endswith(".pdf") else f"{filename}.pdf"


DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
PDF_PATH = DOWNLOAD_DIR / filename_from_url(PDF_URL)

if not PDF_PATH.exists():
    print(f"Downloading {PDF_URL} ...")
    urlretrieve(PDF_URL, PDF_PATH)

print(f"Using PDF: {PDF_PATH.relative_to(REPO_ROOT)}")
print(f"Size:      {PDF_PATH.stat().st_size / 1024 / 1024:.1f} MB")

Using PDF: src/tutorials/t10_vertex_search_agent/downloads/ManualRenta2025Parte1_es_es.pdf
Size:      7.2 MB


## 2. Upload The PDF To Google Cloud Storage

Vertex AI Search imports documents from GCS. We reuse the same staging bucket from t09.

In [3]:
from google.cloud import storage

bucket_name = STAGING_BUCKET.removeprefix("gs://").rstrip("/")
object_name = f"agents-tutorials/vertex-search/{PDF_PATH.name}"

storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(bucket_name)
blob = bucket.blob(object_name)
blob.upload_from_filename(PDF_PATH)

GCS_URI = f"gs://{bucket_name}/{object_name}"
print(f"Uploaded to {GCS_URI}")

Uploaded to gs://edem-practica-mlops-agent-staging/agents-tutorials/vertex-search/ManualRenta2025Parte1_es_es.pdf


## 3. Create Or Reuse A Vertex AI Search Data Store

If `VERTEX_SEARCH_DATA_STORE_ID` is already set in `.env`, the notebook lists existing data stores
to verify it, then reuses it. Otherwise it creates a new unstructured-document data store in the
`global` location (required by Vertex AI Search).

The full resource name looks like:
`projects/MY_PROJECT/locations/global/collections/default_collection/dataStores/MY_DATA_STORE`

In [4]:
from google.cloud import discoveryengine_v1 as discoveryengine

DS_LOCATION = "global"
DS_PARENT = f"projects/{PROJECT_ID}/locations/{DS_LOCATION}/collections/default_collection"

ds_client = discoveryengine.DataStoreServiceClient()

print(f"Existing data stores in {PROJECT_ID}/{DS_LOCATION}:")
all_stores = {ds.name: ds.display_name for ds in ds_client.list_data_stores(parent=DS_PARENT)}
if all_stores:
    for name, display in all_stores.items():
        print(f"  {name}  ({display})")
else:
    print("  (none)")

if EXISTING_DATA_STORE_ID and EXISTING_DATA_STORE_ID in all_stores:
    DATA_STORE_ID = EXISTING_DATA_STORE_ID
    print(f"\nReusing data store: {DATA_STORE_ID}")
else:
    if EXISTING_DATA_STORE_ID:
        print(f"\nWARNING: '{EXISTING_DATA_STORE_ID}' not found — creating a new data store.")

    create_op = ds_client.create_data_store(
        request=discoveryengine.CreateDataStoreRequest(
            parent=DS_PARENT,
            data_store_id=DATA_STORE_SLUG,
            data_store=discoveryengine.DataStore(
                display_name=DATA_STORE_DISPLAY_NAME,
                industry_vertical=discoveryengine.IndustryVertical.GENERIC,
                solution_types=[discoveryengine.SolutionType.SOLUTION_TYPE_SEARCH],
                content_config=discoveryengine.DataStore.ContentConfig.CONTENT_REQUIRED,
            ),
        )
    )
    result = create_op.result(timeout=120)
    DATA_STORE_ID = result.name
    print(f"\nCreated data store: {DATA_STORE_ID}")
    print("\nCopy this into .env:")
    print(f"VERTEX_SEARCH_DATA_STORE_ID={DATA_STORE_ID}")

Existing data stores in edem-practica-mlops/global:
  (none)

Created data store: projects/709491404373/locations/global/collections/default_collection/dataStores/agents-tutorials-search

Copy this into .env:
VERTEX_SEARCH_DATA_STORE_ID=projects/709491404373/locations/global/collections/default_collection/dataStores/agents-tutorials-search


## 4. Import The PDF Into The Data Store

Vertex AI Search reads the PDF from GCS, parses and indexes it. This runs as a long-running
operation and may take a few minutes for a large PDF.

In [5]:
doc_client = discoveryengine.DocumentServiceClient()
branch_parent = f"{DATA_STORE_ID}/branches/default_branch"

import_op = doc_client.import_documents(
    request=discoveryengine.ImportDocumentsRequest(
        parent=branch_parent,
        gcs_source=discoveryengine.GcsSource(
            input_uris=[GCS_URI],
            data_schema="content",
        ),
        reconciliation_mode=(
            discoveryengine.ImportDocumentsRequest.ReconciliationMode.INCREMENTAL
        ),
    )
)

print("Importing documents — this may take a few minutes...")
import_result = import_op.result(timeout=600)
print(f"Import complete.")
if import_result.error_samples:
    for err in import_result.error_samples[:5]:
        print(f"  Error: {err}")

Importing documents — this may take a few minutes...
Import complete.


## 5. Configure And Run The Agent

Copy `VERTEX_SEARCH_DATA_STORE_ID` into `.env` (printed in cell 3 above if you just created it),
then start ADK Web:

```bash
uv run adk web src/tutorials/t10_vertex_search_agent
```

Open http://localhost:8000, select **vertex_search**, and ask questions about the indexed PDF.

> **Note**: Vertex AI Search uses Gemini's model-native grounding — the model retrieves context
> automatically when answering. There is no explicit tool call shown in the conversation trace.